# SETUP — Magenta RT (for `inference`)

This notebook prepares the environment required by the `inference` notebook.

`inference` instantiates the model with `system.MagentaRT(tag="large", lazy=False)`
(the large open-weights model) and streams generated audio to the browser over a
raw-PCM **WebSocket** + `AudioWorklet`, while **Gradio** is used only for parameter
control. The setup below installs exactly what that pipeline needs.

Ensure you are using an A100 GPU or better.

## 1. System Python 3.12

In [ ]:
# Install Python 3.12 (the version Magenta RT / JAX are tested against here).
!sudo apt update
!sudo apt install software-properties-common -y
!sudo add-apt-repository ppa:deadsnakes/ppa -y
!sudo apt install python3.12 python3.12-venv python3.12-dev -y


## 2. Clone the Magenta RT repository

In [ ]:
# Clone the magenta-realtime repo
!git clone https://github.com/magenta/magenta-realtime.git
!cd magenta-realtime


## 3. Create the isolated venv and install Magenta RT (run in the TERMINAL)

Paste the commands below into the **terminal**, not into a Python cell.
All explanatory comments are kept inline so the reasoning travels with the commands.


In [ ]:
# =============================================================================
# SETUP MAGENTA RT on Lightning AI (A100)
# Paste these commands in the TERMINAL (not in a Python cell).
# =============================================================================

# 1. Create a clean venv with python3.12
#    WHY: Lightning AI ships a conda (cloudspace) env with pre-installed
#    jax 0.9.2, flax 0.12.6, numpy 2.x that conflict with each other.
#    An isolated venv gives us full control over the versions.
python3.12 -m venv ~/magenta_venv
source ~/magenta_venv/bin/activate

# 2. Install t5x from the local clone (it is NOT on PyPI)
#    WHY: magenta-rt depends on t5x[gpu], but t5x does not exist on PyPI —
#    it must be installed from the cloned repo. Do this BEFORE magenta-rt,
#    otherwise pip cannot resolve the dependency.
cd /teamspace/studios/this_studio/magenta-realtime/t5x
~/magenta_venv/bin/pip install '.[gpu]'
cd ..

# 3. Install magenta-rt with the GPU dependencies
~/magenta_venv/bin/pip install -e '/teamspace/studios/this_studio/magenta-realtime[gpu]'
~/magenta_venv/bin/pip install tf2jax==0.3.8

# 4. Force the correct jax/flax/numpy/optax/orbax versions
#    WHY: the previous steps pull in versions that are incompatible with one another:
#    - t5x installs jax 0.9.x + numpy 2.x
#    - flax 0.6.x needs jax_config.define_bool_state, absent in jax 0.4.x
#    - flax 0.7.x/0.6.x need jax>=0.7, incompatible with numpy<2
#    - numpy<2 was needed for matplotlib/tensorflow built against numpy 1.x,
#      but the seqio patch (step 5) removes that import, making numpy 2.x fine
#    - optax==0.2.9.dev0 is not on PyPI, so we use 0.2.8 (latest stable)
#    The working combination is: jax 0.9.2 + numpy 2.x + flax 0.12.6
~/magenta_venv/bin/pip install \
  'jax[cuda12]==0.9.2' \
  'numpy>=2' \
  'flax==0.12.6' \
  'orbax-checkpoint==0.11.33' \
  'optax==0.2.8'

~/magenta_venv/bin/pip install 'chex>=0.1.86'

# 5. Patch seqio: removes the tensorflow import from vocabularies.py
#    WHY: seqio imports tensorflow -> keras -> matplotlib, which was built
#    against numpy 1.x and crashed under numpy 2.x. The patch cuts that import
#    chain, fixing the problem at the root.
patch /teamspace/studios/this_studio/magenta_venv/lib/python3.12/site-packages/seqio/vocabularies.py \
  < /teamspace/studios/this_studio/magenta-realtime/patch/seqio_vocabularies.py.patch

# 6. Final check
~/magenta_venv/bin/python -c "import jax; import flax; import seqio; print('dependencies OK')"
~/magenta_venv/bin/python -c "from magenta_rt import system; print('magenta_rt OK')"


## 4. Inference & audio-streaming packages

These are the packages `inference` needs on top of `magenta-rt`.
Each line explains what the library is used for in the pipeline.


In [ ]:
# =============================================================================
# Packages for inference and audio streaming
# =============================================================================

# ipywidgets: enables interactive widgets (sliders, buttons, ...) inside Jupyter;
# without it the Gradio components do not render correctly in the notebook.
!pip install ipywidgets

# nest_asyncio: Magenta RT and the WebSocket servers use asyncio internally, but
# Jupyter already runs its own event loop; this lets the two loops coexist
!pip install nest_asyncio

# gradio: UI control panel only — text/audio prompts and sampling sliders
# (temperature, top-k, guidance). Audio does NOT go through Gradio; it is streamed
# separately over the WebSocket below.
!pip install gradio

# uvicorn <0.30: the ASGI server Gradio runs on. Newer versions are incompatible
# with the way Gradio handles its app lifecycle here.
!pip install "uvicorn<0.30"

# websockets: the core of the real-time audio path. inference runs two
# websockets.serve() servers — one broadcasting raw float32 PCM to the browser
# AudioWorklet ring buffer, one receiving emotion-driven prompt updates.
# It is NOT a magenta-rt dependency, so install it explicitly.
!pip install websockets

# ffmpeg: backend used by librosa/soundfile to decode non-wav audio prompts
# (mp3, ogg). The "Audio prompts" UI accepts .wav .mp3 .ogg, so without ffmpeg
# only .wav would load.
!conda install -y -c conda-forge ffmpeg
